# PCA (Principal Component Analysis)

PCA is unsupervised **dimensionality reduction**: it finds the directions
(principal components) along which the data varies most, and projects onto the
top few — compressing many features into a handful while keeping most of the
information. We use [`linfa-reduction`](https://docs.rs/linfa-reduction).

Like [k-means](../03-clustering/kmeans.ipynb), it works on an unlabelled feature
matrix (`DatasetBase::from`).

In [ ]:
:dep ndarray = { version = "0.15" }
:dep linfa = { version = "0.7" }
:dep linfa-reduction = { version = "0.7" }
use ndarray::array;

// 6 samples, each with 3 correlated features.
let data = array![
    [2.5_f64, 2.4, 1.0],
    [0.5, 0.7, 0.3],
    [2.2, 2.9, 1.2],
    [1.9, 2.2, 0.9],
    [3.1, 3.0, 1.5],
    [2.3, 2.7, 1.1]
];
println!("original: {} samples x {} features", data.nrows(), data.ncols());

In [ ]:
use linfa::prelude::*;
use linfa::DatasetBase;
use linfa_reduction::Pca;
use ndarray::{Array1, Array2};

// Fit PCA keeping 2 components; return the projected points and the share of
// variance each component explains. Explicit types so evcxr persists them.
let (projection, explained): (Array2<f64>, Array1<f64>) = {
    let dataset = DatasetBase::from(data.clone());
    let pca = Pca::params(2).fit(&dataset).expect("PCA fit failed");
    (pca.predict(&dataset), pca.explained_variance_ratio())
};
println!("projected: {} samples x {} components", projection.nrows(), projection.ncols());
println!("explained variance ratio: {:?}", explained);
println!("total variance kept: {:.1}%", explained.sum() * 100.0);

The first component alone captures the bulk of the variance. Plotting the
samples in the new 2-D component space (axis bounds computed from the data):

In [ ]:
:dep plotters = { version = "0.3", default-features = false, features = ["evcxr", "all_series", "all_elements"] }
use plotters::prelude::*;

// Padded axis ranges computed from the projected points. We avoid binding a
// closure to a `let` here: evcxr can't name a closure's type to persist it, so
// we compute the four bounds directly as f64 values instead.
let mut x0 = f64::MAX;
let mut x1 = f64::MIN;
let mut y0 = f64::MAX;
let mut y1 = f64::MIN;
for i in 0..projection.nrows() {
    x0 = x0.min(projection[[i, 0]]);
    x1 = x1.max(projection[[i, 0]]);
    y0 = y0.min(projection[[i, 1]]);
    y1 = y1.max(projection[[i, 1]]);
}
let (x0, x1, y0, y1) = (x0 - 0.5, x1 + 0.5, y0 - 0.5, y1 + 0.5);

evcxr_figure((440, 360), |root| {
    root.fill(&WHITE)?;
    let mut chart = ChartBuilder::on(&root)
        .caption("Samples in PCA space", ("sans-serif", 18))
        .margin(10)
        .x_label_area_size(30)
        .y_label_area_size(40)
        .build_cartesian_2d(x0..x1, y0..y1)?;
    chart.configure_mesh().x_desc("PC1").y_desc("PC2").draw()?;
    chart.draw_series((0..projection.nrows()).map(|i| {
        Circle::new((projection[[i, 0]], projection[[i, 1]]), 5, GREEN.filled())
    }))?;
    Ok(())
})

That completes the core ML arc. See the [crate
reference](../appendix/crate-reference.md) for a cheat sheet and evcxr
troubleshooting.